# Dimensionality & Unidimensionality Check (Author Items)

Input datasets (from `data/stepwise_cleaned_versions/05_dimensionality_inputs/`):
- `ART_pretest_(for Castano)_EN__dimensionality_input__author_response_matrix.csv` — binary 0/1 matrix (908 × 98 author items)
- `ART_pretest_(for Castano)_EN__dimensionality_input__author_item_key.csv` — item metadata (labels, genres, flags, cITC)
- `ART_pretest_(for Castano)_EN__dimensionality_input__foil_response_matrix.csv` — foil responses (secondary diagnostic)
- `ART_pretest_(for Castano)_EN__dimensionality_input__manifest.csv` — provenance metadata

Goals:
1. **Exploratory dimensionality check**: tetrachoric correlations, eigenvalue decomposition, scree plot with parallel analysis, exploratory factor analysis (1- and 2-factor solutions)
2. **Confirmatory unidimensionality check**: one-factor CFA with `lavaan` (WLSMV estimator for binary indicators), reporting RMSEA, CFI, TLI, SRMR, and standardized loadings
3. **Decision**: determine whether author items are sufficiently unidimensional to justify a unidimensional IRT model

References:
- Moore & Gordon (2015): explored factor structure, treated cleaned ART as practically unidimensional, noted secondary factor driven partly by rare items and guessing
- McCarron & Kuperman (2022): used one-factor CFA, accepted fit as adequate, then proceeded to 2PL IRT models

In [ ]:
library(psych)
library(mirt)
library(lavaan)
library(corrplot)
library(ggplot2)

cat("R version:", paste0(R.version$major, ".", R.version$minor), "\n")
cat("Package versions:\n")
for (p in c("psych", "mirt", "lavaan", "corrplot", "ggplot2")) {
  cat(sprintf("  %-12s %s\n", p, as.character(packageVersion(p))))
}

In [ ]:
PROJECT_ROOT <- normalizePath(file.path("..", ".."), mustWork = TRUE)
DATA_DIR     <- file.path(PROJECT_ROOT, "data", "stepwise_cleaned_versions",
                          "05_dimensionality_inputs")

cat("Project root:", PROJECT_ROOT, "\n")
cat("Data dir:    ", DATA_DIR, "\n\n")

# ── Manifest ──
manifest <- read.csv(file.path(DATA_DIR,
  "ART_pretest_(for Castano)_EN__dimensionality_input__manifest.csv"),
  stringsAsFactors = FALSE)
cat("── Manifest ──\n")
print(manifest)

# ── Author response matrix ──
author_df <- read.csv(file.path(DATA_DIR,
  "ART_pretest_(for Castano)_EN__dimensionality_input__author_response_matrix.csv"),
  stringsAsFactors = FALSE)

author_mat <- as.matrix(author_df[, -1])  # drop participant_id
storage.mode(author_mat) <- "integer"

cat("\n── Author response matrix ──\n")
cat(sprintf("  Dimensions: %d participants × %d items\n", nrow(author_mat), ncol(author_mat)))
cat(sprintf("  Value range: [%d, %d]\n", min(author_mat, na.rm = TRUE), max(author_mat, na.rm = TRUE)))
cat(sprintf("  Missing values: %d\n", sum(is.na(author_mat))))

# ── Author item key ──
item_key <- read.csv(file.path(DATA_DIR,
  "ART_pretest_(for Castano)_EN__dimensionality_input__author_item_key.csv"),
  stringsAsFactors = FALSE)

cat("\n── Author item key ──\n")
cat(sprintf("  Items: %d\n", nrow(item_key)))
cat("  Columns:", paste(names(item_key), collapse = ", "), "\n")
cat("\n  Genre distribution:\n")
print(table(item_key$genre))

# ── Sanity checks ──
expected_n <- as.integer(manifest$value[manifest$field == "n_participants"])
expected_k <- as.integer(manifest$value[manifest$field == "n_author_items"])
stopifnot(nrow(author_mat) == expected_n)
stopifnot(ncol(author_mat) == expected_k)
cat(sprintf("\nSanity check passed: n=%d, k=%d match manifest.\n", expected_n, expected_k))

In [ ]:
endorsement <- colMeans(author_mat, na.rm = TRUE)

cat("── Item endorsement rate summary ──\n")
cat(sprintf("  Mean:   %.3f\n", mean(endorsement)))
cat(sprintf("  Median: %.3f\n", median(endorsement)))
cat(sprintf("  Min:    %.3f  (%s)\n", min(endorsement), names(which.min(endorsement))))
cat(sprintf("  Max:    %.3f  (%s)\n", max(endorsement), names(which.max(endorsement))))
cat(sprintf("  Items < .05: %d\n", sum(endorsement < 0.05)))
cat(sprintf("  Items > .95: %d\n", sum(endorsement > 0.95)))

endorse_df <- data.frame(
  item = names(endorsement),
  rate = as.numeric(endorsement),
  stringsAsFactors = FALSE
)
endorse_df <- merge(endorse_df, item_key[, c("matrix_item_id", "item_label", "genre")],
                    by.x = "item", by.y = "matrix_item_id", all.x = TRUE)
endorse_df <- endorse_df[order(endorse_df$rate), ]

options(repr.plot.width = 10, repr.plot.height = 6)
ggplot(endorse_df, aes(x = reorder(item, rate), y = rate, fill = genre)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = c(0.05, 0.95), linetype = "dashed", color = "red", linewidth = 0.5) +
  coord_flip() +
  labs(title = "Author item endorsement rates",
       x = NULL, y = "Endorsement proportion", fill = "Genre") +
  theme_minimal(base_size = 8) +
  theme(axis.text.y = element_text(size = 5))

## 1. Exploratory Dimensionality Check

For binary checklist items, standard Pearson correlations underestimate the true association.
We compute **tetrachoric correlations** (the assumed latent bivariate-normal correlation behind
each 2×2 table) as the appropriate input for factor analysis.

Steps:
1. Tetrachoric correlation matrix (`psych::tetrachoric`)
2. Eigenvalue decomposition + scree plot with parallel analysis
3. One-factor and two-factor EFA on the tetrachoric matrix
4. (Optional) Exploratory IFA via `mirt`

In [ ]:
cat("Computing tetrachoric correlations (this may take a minute for 98 items)...\n")
tet <- psych::tetrachoric(author_mat)

cat("\n── Tetrachoric correlation matrix summary ──\n")
cat(sprintf("  Dimensions: %d × %d\n", nrow(tet$rho), ncol(tet$rho)))

off_diag <- tet$rho[lower.tri(tet$rho)]
cat(sprintf("  Off-diagonal correlations:\n"))
cat(sprintf("    Mean:   %.4f\n", mean(off_diag)))
cat(sprintf("    Median: %.4f\n", median(off_diag)))
cat(sprintf("    Min:    %.4f\n", min(off_diag)))
cat(sprintf("    Max:    %.4f\n", max(off_diag)))
cat(sprintf("    SD:     %.4f\n", sd(off_diag)))

# Check positive-definiteness
eig_vals_raw <- eigen(tet$rho, only.values = TRUE)$values
n_negative <- sum(eig_vals_raw < 0)
if (n_negative > 0) {
  cat(sprintf("\n  WARNING: Matrix is NOT positive definite (%d negative eigenvalues).\n", n_negative))
  cat("  Will use smoothed version where needed.\n")
} else {
  cat("\n  Matrix is positive definite.\n")
}

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)
corrplot::corrplot(
  tet$rho,
  method    = "color",
  order     = "hclust",
  hclust.method = "ward.D2",
  tl.cex    = 0.35,
  tl.col    = "black",
  cl.cex    = 0.7,
  title     = "Tetrachoric correlation matrix (hierarchical clustering order)",
  mar       = c(0, 0, 2, 0)
)

In [ ]:
eig_vals <- eigen(tet$rho)$values

cat("── First 15 eigenvalues of tetrachoric correlation matrix ──\n")
eig_table <- data.frame(
  Factor      = 1:15,
  Eigenvalue  = round(eig_vals[1:15], 4),
  Prop_Var    = round(eig_vals[1:15] / sum(eig_vals[eig_vals > 0]), 4),
  Cum_Var     = round(cumsum(eig_vals[1:15]) / sum(eig_vals[eig_vals > 0]), 4)
)
print(eig_table, row.names = FALSE)

cat(sprintf("\nRatio eigenvalue1 / eigenvalue2: %.2f\n", eig_vals[1] / eig_vals[2]))
cat(sprintf("Eigenvalue 1: %.4f  (%.1f%% of variance)\n",
    eig_vals[1], 100 * eig_vals[1] / sum(eig_vals[eig_vals > 0])))

# Scree plot
options(repr.plot.width = 9, repr.plot.height = 5)
eig_plot_df <- data.frame(Factor = 1:length(eig_vals), Eigenvalue = eig_vals)

ggplot(eig_plot_df[1:20, ], aes(x = Factor, y = Eigenvalue)) +
  geom_line(linewidth = 0.8, color = "steelblue") +
  geom_point(size = 2.5, color = "steelblue") +
  geom_hline(yintercept = 1.0, linetype = "dashed", color = "red") +
  annotate("text", x = 18, y = 1.15, label = "Kaiser criterion (eigenvalue = 1)",
           size = 3, color = "red") +
  annotate("text", x = 3, y = eig_vals[1] - 0.5,
           label = sprintf("Ratio λ1/λ2 = %.2f", eig_vals[1] / eig_vals[2]),
           size = 3.5, fontface = "bold") +
  scale_x_continuous(breaks = 1:20) +
  labs(title = "Scree plot (tetrachoric correlation matrix)",
       x = "Factor number", y = "Eigenvalue") +
  theme_minimal(base_size = 11)

In [ ]:
cat("── Parallel analysis (tetrachoric, may take a few minutes) ──\n\n")
options(repr.plot.width = 9, repr.plot.height = 5)
pa <- psych::fa.parallel(
  author_mat,
  cor   = "tet",
  fa    = "fa",
  n.iter = 20,
  main  = "Parallel analysis scree plot (tetrachoric)"
)

cat(sprintf("\nParallel analysis suggests %d factor(s).\n", pa$nfact))
cat("Actual vs simulated 95th-percentile eigenvalues (first 10):\n")
comp_df <- data.frame(
  Factor    = 1:min(10, length(pa$fa.values)),
  Actual    = round(pa$fa.values[1:min(10, length(pa$fa.values))], 4),
  Simulated = round(pa$fa.sim[1:min(10, length(pa$fa.sim))], 4)
)
print(comp_df, row.names = FALSE)

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  ONE-FACTOR EFA (tetrachoric, minres)\n")
cat("══════════════════════════════════════════════════════════════\n\n")

fa1 <- psych::fa(tet$rho, nfactors = 1, n.obs = nrow(author_mat), fm = "minres")
print(fa1)

# Loadings table with item metadata
load1 <- data.frame(
  item    = rownames(fa1$loadings),
  loading = as.numeric(fa1$loadings[, 1]),
  communality = fa1$communalities,
  uniqueness  = fa1$uniquenesses,
  stringsAsFactors = FALSE
)
load1 <- merge(load1, item_key[, c("matrix_item_id", "item_label", "genre",
               "selection_rate_pct", "corrected_item_total_corr")],
               by.x = "item", by.y = "matrix_item_id", all.x = TRUE)
load1 <- load1[order(load1$loading), ]

cat("\n── Items with lowest loadings (< 0.30) ──\n")
low_load <- load1[load1$loading < 0.30, ]
if (nrow(low_load) > 0) {
  print(low_load[, c("item", "item_label", "genre", "loading", "selection_rate_pct")],
        row.names = FALSE)
} else {
  cat("  None — all loadings >= 0.30\n")
}

cat(sprintf("\nProportion variance explained (1 factor): %.4f\n", fa1$Vaccounted[2, 1]))
cat(sprintf("RMSR: %.4f\n", fa1$rms))

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  TWO-FACTOR EFA (tetrachoric, minres, oblimin rotation)\n")
cat("══════════════════════════════════════════════════════════════\n\n")

fa2 <- psych::fa(tet$rho, nfactors = 2, n.obs = nrow(author_mat),
                 fm = "minres", rotate = "oblimin")
print(fa2)

load2 <- data.frame(
  item = rownames(fa2$loadings),
  F1   = as.numeric(fa2$loadings[, 1]),
  F2   = as.numeric(fa2$loadings[, 2]),
  stringsAsFactors = FALSE
)
load2 <- merge(load2, item_key[, c("matrix_item_id", "item_label", "genre", "selection_rate_pct")],
               by.x = "item", by.y = "matrix_item_id", all.x = TRUE)

cat("\n── Factor correlation ──\n")
print(round(fa2$Phi, 4))

cat("\n── Items loading primarily on Factor 2 (|F2| > |F1|) ──\n")
f2_primary <- load2[abs(load2$F2) > abs(load2$F1), ]
f2_primary <- f2_primary[order(-abs(f2_primary$F2)), ]
if (nrow(f2_primary) > 0) {
  print(f2_primary[, c("item", "item_label", "genre", "F1", "F2", "selection_rate_pct")],
        row.names = FALSE)
} else {
  cat("  None — all items load primarily on Factor 1\n")
}

cat(sprintf("\nCumulative variance (2 factors): %.4f\n",
    sum(fa2$Vaccounted[2, ])))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 12)

load1_sorted <- load1[order(load1$loading), ]
load1_sorted$item_label_short <- ifelse(
  nchar(load1_sorted$item_label) > 25,
  paste0(substr(load1_sorted$item_label, 1, 22), "..."),
  load1_sorted$item_label
)
load1_sorted$display <- paste0(load1_sorted$item, ": ", load1_sorted$item_label_short)
load1_sorted$display <- factor(load1_sorted$display,
                               levels = load1_sorted$display)

ggplot(load1_sorted, aes(x = loading, y = display, color = genre)) +
  geom_point(size = 2) +
  geom_segment(aes(xend = 0, yend = display), linewidth = 0.3) +
  geom_vline(xintercept = 0.30, linetype = "dashed", color = "red", linewidth = 0.4) +
  annotate("text", x = 0.32, y = 5, label = "0.30 threshold",
           size = 2.8, color = "red", hjust = 0) +
  labs(title = "One-factor EFA: standardized loadings (tetrachoric, minres)",
       x = "Factor loading", y = NULL, color = "Genre") +
  theme_minimal(base_size = 8) +
  theme(axis.text.y = element_text(size = 5),
        legend.position = "bottom")

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  EXPLORATORY IFA WITH mirt (unidimensional 2PL)\n")
cat("══════════════════════════════════════════════════════════════\n\n")

cat("Fitting unidimensional 2PL model (EM algorithm)...\n")
mod1 <- mirt::mirt(author_mat, model = 1, itemtype = "2PL", verbose = FALSE)

cat("\n── Model summary ──\n")
print(mod1)

cat("\n── Item parameters (slope a1, intercept d) ──\n")
item_pars <- coef(mod1, IRTpars = TRUE, simplify = TRUE)$items
item_pars_df <- as.data.frame(item_pars)
item_pars_df$item <- rownames(item_pars_df)
item_pars_df <- merge(item_pars_df,
                      item_key[, c("matrix_item_id", "item_label", "genre")],
                      by.x = "item", by.y = "matrix_item_id", all.x = TRUE)
item_pars_df <- item_pars_df[order(item_pars_df$a), ]
print(item_pars_df, row.names = FALSE)

cat("\n── Items with near-zero discrimination (a < 0.30) ──\n")
low_disc <- item_pars_df[item_pars_df$a < 0.30, ]
if (nrow(low_disc) > 0) {
  print(low_disc[, c("item", "item_label", "genre", "a", "b")], row.names = FALSE)
} else {
  cat("  None — all items have a >= 0.30\n")
}

In [ ]:
cat("── M2 limited-information goodness-of-fit (may take a moment) ──\n\n")
m2_fit <- tryCatch(
  mirt::M2(mod1, type = "C2"),
  error = function(e) {
    cat("M2 computation failed:", conditionMessage(e), "\n")
    cat("Trying M2* (M2 with type='C2' fallback)...\n")
    tryCatch(mirt::M2(mod1), error = function(e2) {
      cat("M2 also failed:", conditionMessage(e2), "\n")
      NULL
    })
  }
)

if (!is.null(m2_fit)) {
  print(m2_fit)
} else {
  cat("M2 fit statistics could not be computed (common with large item sets).\n")
  cat("Proceeding with CFA-based fit assessment instead.\n")
}

## 2. Confirmatory Unidimensionality Check

Following McCarron & Kuperman (2022), we fit a **one-factor CFA** treating all author items as
ordered/categorical indicators, using the **WLSMV** (weighted least squares, mean and variance
adjusted) estimator — the standard choice for binary/ordinal data in `lavaan`.

Reporting:
- RMSEA (+ 90% CI)
- CFI
- TLI
- SRMR (if available)
- Standardized factor loadings

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  ONE-FACTOR CFA (lavaan, WLSMV, categorical indicators)\n")
cat("══════════════════════════════════════════════════════════════\n\n")

author_cfa_df <- as.data.frame(author_mat)
item_names    <- colnames(author_cfa_df)

model_syntax <- paste0("F1 =~ ", paste(item_names, collapse = " + "))
cat("Model syntax (first 120 chars):", substr(model_syntax, 1, 120), "...\n\n")

cat("Fitting CFA (this may take several minutes with 98 binary items)...\n")
cfa_fit <- lavaan::cfa(
  model     = model_syntax,
  data      = author_cfa_df,
  ordered   = item_names,
  estimator = "WLSMV"
)

cat("\n── Full CFA summary ──\n\n")
summary(cfa_fit, fit.measures = TRUE, standardized = TRUE)

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  FIT INDICES SUMMARY\n")
cat("══════════════════════════════════════════════════════════════\n\n")

fm <- lavaan::fitMeasures(cfa_fit, c(
  "chisq", "df", "pvalue",
  "rmsea", "rmsea.ci.lower", "rmsea.ci.upper", "rmsea.pvalue",
  "cfi", "tli",
  "srmr"
))

fit_table <- data.frame(
  Index = c("Chi-square", "df", "p-value",
            "RMSEA", "RMSEA 90% CI lower", "RMSEA 90% CI upper", "RMSEA close-fit p",
            "CFI", "TLI",
            "SRMR"),
  Value = round(fm, 4),
  Threshold = c("", "", "",
                "< 0.08 acceptable, < 0.06 good", "", "", "> 0.05",
                "> 0.90 acceptable, > 0.95 good",
                "> 0.90 acceptable, > 0.95 good",
                "< 0.08 acceptable"),
  stringsAsFactors = FALSE
)
print(fit_table, row.names = FALSE)

cat("\n── Quick assessment ──\n")
cat(sprintf("  RMSEA = %.4f [%.4f, %.4f]  →  %s\n",
    fm["rmsea"], fm["rmsea.ci.lower"], fm["rmsea.ci.upper"],
    ifelse(fm["rmsea"] < 0.06, "GOOD", ifelse(fm["rmsea"] < 0.08, "ACCEPTABLE", "POOR"))))
cat(sprintf("  CFI   = %.4f  →  %s\n",
    fm["cfi"],
    ifelse(fm["cfi"] > 0.95, "GOOD", ifelse(fm["cfi"] > 0.90, "ACCEPTABLE", "POOR"))))
cat(sprintf("  TLI   = %.4f  →  %s\n",
    fm["tli"],
    ifelse(fm["tli"] > 0.95, "GOOD", ifelse(fm["tli"] > 0.90, "ACCEPTABLE", "POOR"))))
if (!is.na(fm["srmr"])) {
  cat(sprintf("  SRMR  = %.4f  →  %s\n",
      fm["srmr"],
      ifelse(fm["srmr"] < 0.08, "ACCEPTABLE", "POOR")))
}

In [ ]:
cat("── Standardized factor loadings (CFA) ──\n\n")

std_est <- lavaan::standardizedSolution(cfa_fit)
loadings_df <- std_est[std_est$op == "=~", c("rhs", "est.std", "se", "z", "pvalue")]
names(loadings_df) <- c("item", "std_loading", "se", "z", "pvalue")

loadings_df <- merge(loadings_df,
                     item_key[, c("matrix_item_id", "item_label", "genre", "selection_rate_pct")],
                     by.x = "item", by.y = "matrix_item_id", all.x = TRUE)
loadings_df <- loadings_df[order(loadings_df$std_loading), ]

cat(sprintf("  Items total:     %d\n", nrow(loadings_df)))
cat(sprintf("  Loading >= 0.30: %d  (%.1f%%)\n",
    sum(loadings_df$std_loading >= 0.30),
    100 * sum(loadings_df$std_loading >= 0.30) / nrow(loadings_df)))
cat(sprintf("  Loading <  0.30: %d\n", sum(loadings_df$std_loading < 0.30)))
cat(sprintf("  Loading <  0.20: %d\n", sum(loadings_df$std_loading < 0.20)))
cat(sprintf("  Mean loading:    %.4f\n", mean(loadings_df$std_loading)))
cat(sprintf("  Median loading:  %.4f\n", median(loadings_df$std_loading)))

cat("\n── Full loadings table (sorted ascending) ──\n")
print(loadings_df[, c("item", "item_label", "genre", "std_loading", "se",
                      "selection_rate_pct")],
      row.names = FALSE)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 12)

loadings_df$item_label_short <- ifelse(
  nchar(loadings_df$item_label) > 25,
  paste0(substr(loadings_df$item_label, 1, 22), "..."),
  loadings_df$item_label
)
loadings_df$display <- paste0(loadings_df$item, ": ", loadings_df$item_label_short)
loadings_df$display <- factor(loadings_df$display, levels = loadings_df$display)

ggplot(loadings_df, aes(x = std_loading, y = display, fill = genre)) +
  geom_col(width = 0.6) +
  geom_vline(xintercept = 0.30, linetype = "dashed", color = "red", linewidth = 0.4) +
  annotate("text", x = 0.32, y = 5, label = "0.30 threshold",
           size = 2.8, color = "red", hjust = 0) +
  labs(title = "One-factor CFA: standardized loadings (WLSMV)",
       x = "Standardized loading", y = NULL, fill = "Genre") +
  theme_minimal(base_size = 8) +
  theme(axis.text.y = element_text(size = 5),
        legend.position = "bottom")

## 3. Foil Diagnostic (Secondary)

Foils are examined separately as a response-style diagnostic set. They should **not** be mixed
with author items in the main latent-trait model for print exposure.

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  FOIL RESPONSE-STYLE DIAGNOSTIC\n")
cat("══════════════════════════════════════════════════════════════\n\n")

foil_df <- read.csv(file.path(DATA_DIR,
  "ART_pretest_(for Castano)_EN__dimensionality_input__foil_response_matrix.csv"),
  stringsAsFactors = FALSE)
foil_mat <- as.matrix(foil_df[, -1])
storage.mode(foil_mat) <- "integer"

cat(sprintf("Foil matrix: %d participants × %d items\n", nrow(foil_mat), ncol(foil_mat)))

foil_endorsement <- colMeans(foil_mat, na.rm = TRUE)
cat(sprintf("Mean foil endorsement: %.4f\n", mean(foil_endorsement)))
cat(sprintf("Max foil endorsement:  %.4f\n", max(foil_endorsement)))

# Remove zero-variance foils before tetrachoric
foil_var <- apply(foil_mat, 2, var, na.rm = TRUE)
foil_mat_clean <- foil_mat[, foil_var > 0]
cat(sprintf("Foils with non-zero variance: %d / %d\n", ncol(foil_mat_clean), ncol(foil_mat)))

cat("\nComputing foil tetrachoric correlations...\n")
foil_tet <- tryCatch(
  psych::tetrachoric(foil_mat_clean),
  error = function(e) {
    cat("Tetrachoric computation failed:", conditionMessage(e), "\n")
    NULL
  }
)

if (!is.null(foil_tet)) {
  foil_eig <- eigen(foil_tet$rho)$values
  cat("\nFirst 10 eigenvalues (foil tetrachoric):\n")
  print(round(foil_eig[1:min(10, length(foil_eig))], 4))
  cat(sprintf("Ratio eigenvalue1/eigenvalue2: %.2f\n", foil_eig[1] / foil_eig[2]))

  cat("\nNote: Foils are NOT included in the main latent-trait model.\n")
  cat("This analysis is for response-style diagnostics only.\n")
}

## 4. Decision & Summary

**Decision criteria** (following Moore & Gordon 2015; McCarron & Kuperman 2022):

Proceed with **unidimensional IRT** if:
1. The one-factor CFA fit is acceptable (RMSEA < 0.08, CFI > 0.90)
2. Most loadings are substantial (>= 0.30)
3. Any secondary factor mostly reflects extremely rare items or guessing-related items
4. The construct remains interpretable as *print exposure / author recognition*

Moore & Gordon saw evidence of a second factor but interpreted it as driven partly by rare items
and criterion-shift/guessing behavior, then continued with a cleaned unidimensional item set.

In [ ]:
cat("══════════════════════════════════════════════════════════════\n")
cat("  UNIDIMENSIONALITY DECISION SUMMARY\n")
cat("══════════════════════════════════════════════════════════════\n\n")

# Gather evidence
n_items <- nrow(loadings_df)
n_above_30 <- sum(loadings_df$std_loading >= 0.30)
n_below_20 <- sum(loadings_df$std_loading < 0.20)
pct_above_30 <- 100 * n_above_30 / n_items

rmsea_val <- fm["rmsea"]
cfi_val   <- fm["cfi"]
tli_val   <- fm["tli"]

eig_ratio <- eig_vals[1] / eig_vals[2]

pa_nfact <- pa$nfact

# Print summary
cat("── Evidence from exploratory analyses ──\n")
cat(sprintf("  Eigenvalue ratio (λ1/λ2):           %.2f\n", eig_ratio))
cat(sprintf("  Parallel analysis suggested factors: %d\n", pa_nfact))
cat(sprintf("  1-factor EFA variance explained:     %.1f%%\n",
    100 * fa1$Vaccounted[2, 1]))

cat("\n── Evidence from confirmatory CFA ──\n")
cat(sprintf("  RMSEA: %.4f  %s\n", rmsea_val,
    ifelse(rmsea_val < 0.06, "[GOOD]", ifelse(rmsea_val < 0.08, "[ACCEPTABLE]", "[POOR]"))))
cat(sprintf("  CFI:   %.4f  %s\n", cfi_val,
    ifelse(cfi_val > 0.95, "[GOOD]", ifelse(cfi_val > 0.90, "[ACCEPTABLE]", "[POOR]"))))
cat(sprintf("  TLI:   %.4f  %s\n", tli_val,
    ifelse(tli_val > 0.95, "[GOOD]", ifelse(tli_val > 0.90, "[ACCEPTABLE]", "[POOR]"))))

cat("\n── Loading quality ──\n")
cat(sprintf("  Items with loading >= 0.30: %d / %d  (%.1f%%)\n", n_above_30, n_items, pct_above_30))
cat(sprintf("  Items with loading <  0.20: %d\n", n_below_20))
cat(sprintf("  Mean loading:              %.4f\n", mean(loadings_df$std_loading)))

# Items on secondary factor
cat("\n── Secondary factor interpretation ──\n")
if (exists("f2_primary") && nrow(f2_primary) > 0) {
  cat(sprintf("  %d items loaded primarily on Factor 2 in the 2-factor EFA.\n", nrow(f2_primary)))
  f2_rates <- as.numeric(f2_primary$selection_rate_pct)
  cat(sprintf("  Their mean endorsement rate: %.1f%%\n", mean(f2_rates, na.rm = TRUE)))
  cat(sprintf("  Their endorsement range:     %.1f%% – %.1f%%\n",
      min(f2_rates, na.rm = TRUE), max(f2_rates, na.rm = TRUE)))
  cat("  Genre composition:\n")
  print(table(f2_primary$genre))
} else {
  cat("  No items loaded primarily on Factor 2.\n")
}

# Final recommendation
cat("\n══════════════════════════════════════════════════════════════\n")
cfa_acceptable <- (rmsea_val < 0.08) && (cfi_val > 0.90)
loadings_ok    <- pct_above_30 >= 80

if (cfa_acceptable && loadings_ok) {
  cat("  RECOMMENDATION: PROCEED with unidimensional IRT.\n")
  cat("  The one-factor model shows acceptable fit and the large\n")
  cat("  majority of items have substantial loadings.\n")
} else if (cfa_acceptable) {
  cat("  RECOMMENDATION: PROCEED with CAUTION.\n")
  cat("  CFA fit is acceptable, but a notable proportion of items\n")
  cat("  have weak loadings. Consider removing items < 0.20 before IRT.\n")
} else {
  cat("  RECOMMENDATION: INVESTIGATE FURTHER before proceeding.\n")
  cat("  CFA fit indices do not meet standard acceptability thresholds.\n")
  cat("  Consider item removal, bifactor models, or multidimensional IRT.\n")
}
cat("══════════════════════════════════════════════════════════════\n")

if (n_below_20 > 0) {
  cat("\n── Items flagged for potential removal (loading < 0.20) ──\n")
  flagged <- loadings_df[loadings_df$std_loading < 0.20,
                         c("item", "item_label", "genre", "std_loading", "selection_rate_pct")]
  print(flagged, row.names = FALSE)
}